In [ ]:
# Install dependencies
!pip install -q torch transformers[torch] datasets accelerate scikit-learn pandas numpy matplotlib seaborn shap tqdm

In [ ]:
# ============ SETUP: imports, plotting, figure dir ============
import os, sys, json, time, re, warnings, zipfile, shutil, random, csv
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             average_precision_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import label_binarize
import torch
from google.colab import files

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 200

FIG_DIR = '/content/figures'
os.makedirs(FIG_DIR, exist_ok=True)

LABEL_MAP = {0: 'Legitimate Financial Communication',
             1: 'Traditional Phishing',
             2: 'AI-Generated Phishing'}
CLASS_NAMES = ['Legitimate', 'Traditional Phishing', 'AI-Generated Phishing']
CLASS_COLORS = ['#2ecc71', '#f39c12', '#e74c3c']
print('Setup complete. PyTorch:', torch.__version__)

In [ ]:

# ==================== DATASET GENERATION (embedded) ====================
# This produces the identical 5000-sample dataset as dataset/generate_dataset.py
import csv, os, random
from typing import List, Tuple

random.seed(42)

BANKS = [
    "GTBank", "UBA", "Access Bank", "Zenith Bank", "Fidelity Bank",
    "First Bank", "Moniepoint", "PalmPay", "Opay", "Stanbic IBTC",
    "Ecobank", "Union Bank", "Wema Bank", "Sterling Bank", "Polaris Bank",
    "Keystone Bank", "FCMB", "SunTrust Bank", "Providus Bank", "TajBank",
]
FIN_TECH = [
    "Moniepoint", "PalmPay", "Opay", "Kuda Bank", "Carbon",
    "FairMoney", "Branch", "ALAT by Wema", "VBank", "Mint",
    "Chipper Cash", "Flutterwave", "Paystack", "Interswitch",
]
ORGS = BANKS + FIN_TECH

LEGIT_TEMPLATES = [
    "Dear {customer}, a debit of NGN{amount} was made on your {bank} account "
    "{account} at {merchant} on {date}. Available balance: NGN{balance}. If not you, call {phone}.",
    "Debit Alert: NGN{amount} spent at {merchant} on {date} from {bank} account "
    "{account}. Balance: NGN{balance}.",
    "Transaction alert: Withdrawal of NGN{amount} at ATM {atm_id} on {date}. "
    "{bank} account {account} balance: NGN{balance}. Thank you.",
    "Credit Alert: NGN{amount} received from {sender} into your {bank} account "
    "{account} on {date}. Balance: NGN{balance}. Thank you for banking with us.",
    "Your transfer of NGN{amount} to {recipient} on {date} was successful. "
    "Reference: {ref}. {bank} account {account} balance: NGN{balance}.",
    "Salary payment of NGN{amount} credited to your {bank} account {account} "
    "on {date}. Balance: NGN{balance}. Regards, {bank}.",
    "We received a password reset request for your {bank} account. "
    "Click here to reset: {link}. If you did not request this, ignore this message.",
    "Your {bank} account password was changed successfully on {date}. "
    "If you did not authorise this, contact {phone} immediately.",
    "Kindly update your KYC details to continue enjoying seamless banking. "
    "Visit any {bank} branch or click {link} to update. Reference: {ref}.",
    "Your BVN has been linked to your {bank} account successfully. "
    "Thank you for your cooperation.",
    "Your monthly statement for {bank} account {account} is ready. "
    "Download at {link}. Password: your birthdate.",
    "Your {bank} account {account} has been credited with NGN{amount} "
    "being the sum of your monthly savings. Balance: NGN{balance}. Keep saving!",
    "Your {bank} debit card will expire on {date}. A new card will be "
    "delivered to your branch within 5 business days.",
    "Cardless withdrawal: Use code {ref} at any {bank} ATM to withdraw "
    "up to NGN{amount}. Valid for 1 hour.",
    "Dial *737# to transfer, buy airtime, or pay bills with your {bank} account.",
    "Your {bank} mobile app login was detected from a new device. "
    "If this was you, no action needed. Otherwise contact {phone}.",
    "Congratulations! You are pre-qualified for a {bank} salary loan of up to "
    "NGN{amount} at {rate}% interest. Reply YES to opt in. T&C apply.",
    "Your {bank} credit card application has been approved. "
    "Your card will be delivered within 5 working days.",
]

TRAD_PHISH_TEMPLATES = [
    "URGENT!!! Your BVN has been BLOCKED. Click here to verify now: {link}",
    "Dear Customer, your account will be SUSPENDED if you don't update your "
    "details now. Click: {link}",
    "Alert: NGN{amount} deducted from your account. If not you, call {phone} "
    "immediately or click {link} to reverse.",
    "Your ATM card has been deactivated. Update your PIN here: {link}",
    "Security Alert!!! Unusual login detected. Verify your account: {link}",
    "Your {bank} account requires immediate reactivation. "
    "Click here to reactivate: {link}",
    "Congratulations! You won NGN{amount} in our promotion. "
    "Claim your prize: {link}",
    "Your NIN must be linked to your BVN immediately or your account will be "
    "frozen. Verify now: {link}",
    "Dear {customer}, your internet banking has been locked. "
    "Unlock here: {link}",
    "Warning: Your account has been flagged for suspicious activity. "
    "Confirm your identity: {link}",
    "You have a pending refund of NGN{amount}. Process: {link}",
    "Your account has been credited with NGN{amount} by mistake. "
    "Return the money: {link}",
    "Dear Customer, update your account to continue enjoying our service. "
    "Click: {link}",
    "You have been selected for a loan of NGN{amount}. "
    "Accept now: {link}",
    "Your card has been charged NGN{amount} for Netflix. "
    "If not you, dispute: {link}",
    "Account upgrade available! Click to upgrade your account: {link}",
    "Your BVN has expired! Update your BVN: {link}",
    "Payment of NGN{amount} failed. Update your account: {link}",
    "Dear customer, your bank details are required for verification. "
    "Send to this email: {email}",
    "Your account will be debited NGN{amount} monthly. Cancel: {link}",
    "Immediate action required: Confirm your account details: {link}",
    "You have {count} unapproved transactions. Approve: {link}",
    "Your online banking access has been restricted. "
    "Restore access: {link}",
    "Your account has been compromised! Secure it here: {link}",
]

AI_PHISH_TEMPLATES = [
    "Subject: Mandatory BVN-NIN Linkage Compliance Notice\n\n"
    "Dear {customer},\n\n"
    "This is to notify you that the Central Bank of Nigeria (CBN) now requires "
    "all bank accounts to have their BVN linked to the National Identification Number (NIN) "
    "by {date}. Accounts not complying will be placed on restricted status.\n\n"
    "To complete the linkage securely, please visit: {link}\n\n"
    "This process takes less than 2 minutes.\n\n"
    "Thank you for your cooperation.\n"
    "Compliance Department\n{bank}",
    "Subject: Account Suspension Notice \u2013 Immediate Action Required\n\n"
    "Dear {customer},\n\n"
    "Our fraud detection system has identified multiple failed login attempts on your "
    "{bank} account. As a precautionary measure, your account has been temporarily "
    "suspended.\n\n"
    "To restore full access, you must verify your identity by completing the "
    "verification form at: {link}\n\n"
    "Failure to do so within 24 hours will result in permanent account closure.\n\n"
    "Regards,\n"
    "Security Operations Center\n{bank}",
    "Subject: Unauthorised Transaction Detected \u2013 Reversal Required\n\n"
    "Dear {customer},\n\n"
    "A transaction of NGN{amount} was initiated from your {bank} account "
    "to a beneficiary not on your approved list. If you did not authorise this "
    "transaction, please initiate a reversal within the next 12 hours.\n\n"
    "Reversal link: {link}\n\n"
    "Please note that failure to act may lead to permanent loss of funds.\n\n"
    "Sincerely,\n"
    "Fraud Investigation Unit\n{bank}",
    "Subject: KYC Update \u2013 Final Notice\n\n"
    "Dear {customer},\n\n"
    "This is your final reminder to update your Know Your Customer (KYC) "
    "information with {bank}. As mandated by the Central Bank of Nigeria, all "
    "accounts with incomplete KYC records will be restricted effective {date}.\n\n"
    "Update your information securely at: {link}\n\n"
    "Thank you for banking with us.\n\n"
    "Best regards,\n"
    "Customer Relationship Management\n{bank}",
    "Subject: Debit Card Replacement Notification\n\n"
    "Dear {customer},\n\n"
    "Your {bank} debit card has been flagged for replacement due to a "
    "potential security vulnerability. To avoid interruption of service, please "
    "confirm your delivery address and activate your new card using the secure "
    "portal below.\n\n"
    "Secure portal: {link}\n\n"
    "Your new card will be delivered within 3-5 business days.\n\n"
    "Warm regards,\n"
    "Card Services\n{bank}",
    "Subject: CBN Directive \u2013 Beneficiary Account Verification\n\n"
    "Dear {customer},\n\n"
    "Following the Central Bank of Nigeria's revised guidelines on electronic "
    "banking, all beneficiaries on your {bank} account must be re-verified "
    "to ensure compliance with anti-money laundering regulations.\n\n"
    "Please verify your beneficiaries at: {link}\n\n"
    "Unverified beneficiaries will be removed from your profile after {date}.\n\n"
    "Yours faithfully,\n"
    "Compliance and Risk Management\n{bank}",
    "Subject: Account Restriction Warning \u2013 Regulatory Compliance\n\n"
    "Dear {customer},\n\n"
    "Your {bank} account is scheduled for restriction due to incomplete "
    "regulatory documentation. To prevent this, please submit the required "
    "documents through our secure upload portal.\n\n"
    "Upload portal: {link}\n\n"
    "This restriction will take effect in 48 hours if not addressed.\n\n"
    "Sincerely,\n"
    "Regulatory Compliance Team\n{bank}",
    "Subject: Refund of Duplicate Charges\n\n"
    "Dear {customer},\n\n"
    "Our audit team has identified that your {bank} account was incorrectly "
    "charged NGN{amount} due to a system error. You are entitled to a full "
    "refund of this amount.\n\n"
    "To process your refund, please confirm your account details at: {link}\n\n"
    "We sincerely apologise for the inconvenience.\n\n"
    "Best regards,\n"
    "Audit and Reconciliation\n{bank}",
    "Subject: Enhanced Security Verification Required\n\n"
    "Dear {customer},\n\n"
    "As part of our ongoing commitment to protecting your financial assets, "
    "{bank} is implementing an enhanced security protocol for all online "
    "banking users.\n\n"
    "You are required to complete a multi-factor authentication setup by "
    "visiting: {link}\n\n"
    "Accounts without the enhanced security enabled will be restricted from "
    "online banking after {date}.\n\n"
    "Thank you for your understanding.\n\n"
    "Yours sincerely,\n"
    "Information Security Team\n{bank}",
    "Subject: Notification of Beneficial Ownership Declaration\n\n"
    "Dear {customer},\n\n"
    "In accordance with the Companies and Allied Matters Act (CAMA) and CBN "
    "regulations, all corporate account holders must submit a beneficial "
    "ownership declaration.\n\n"
    "File your declaration securely at: {link}\n\n"
    "Non-compliance will result in account restriction after {date}.\n\n"
    "Regards,\n"
    "Corporate Banking Division\n{bank}",
    "Subject: Pending Tax Compliance Verification\n\n"
    "Dear {customer},\n\n"
    "The Federal Inland Revenue Service (FIRS) has requested that all financial "
    "institutions verify the tax identification numbers (TIN) of their customers.\n\n"
    "Please submit your TIN for verification at: {link}\n\n"
    "Accounts with unverified TINs may be subject to withholding tax deductions "
    "at source.\n\n"
    "Thank you for your cooperation.\n\n"
    "Best regards,\n"
    "Tax Compliance Unit\n{bank}",
]

CUSTOMER_NAMES = [
    "Chidi Okonkwo", "Aisha Bello", "Emeka Okafor", "Funke Adebayo",
    "Segun Ogunlade", "Ngozi Eze", "Tunde Balogun", "Fatima Usman",
    "Oluwaseun Adeyemi", "Chioma Nwosu", "Ibrahim Danjuma", "Yetunde Lawal",
    "Uchenna Obi", "Temitope Ojo", "Grace Okoro", "Kayode Adewale",
    "Halima Abubakar", "Ebuka Nwachukwu", "Ronke Adedeji", "Musa Kuti",
    "Akintunde Ogunbiyi", "Chinaza Ezeh", "Bamidele Ogun", "Folake Abiola",
    "Ifeanyi Okoro", "Kemi Alabi", "Olumide Fasanya", "Sade Bamidele",
    "Tanko Yaro", "Zainab Abdullah",
]
MERCHANTS = [
    "ShopRite", "Jumia", "Konga", "Total Energies", "MRS Oil",
    "Shell", "SPAR", "Hubmart", "Justrite", "Addide",
    "Chicken Republic", "Mr Biggs", "Domino's Pizza", "KFC",
    "Cold Stone", "Payporte", "Slot", "Pointek", "Amazon",
    "Netflix", "Spotify", "Google Play", "Apple Store", "DStv",
    "GoTV", "Startimes", "Interswitch", "Remita", "Bet9ja", "SportyBet",
]
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
ACCOUNT_NUMS = [
    "0123456789", "9876543210", "1234509876", "6789012345",
    "1112223334", "4445556667", "7778889990", "2223334445",
    "5556667778", "8889990001", "3334445556", "6667778889",
    "9990001112", "0001112223", "7890123456",
]


def _rand_amount(min_v=500, max_v=5000000):
    return random.randint(min_v, max_v)


def _rand_balance(amount):
    return amount + random.randint(1000, 50000000)


def _rand_date():
    day = random.randint(1, 28)
    month = random.choice(MONTHS)
    year = random.choice(["2024", "2025", "2026"])
    return f"{day}-{month}-{year}"


def _rand_ref():
    return random.choice([
        "TXN" + str(random.randint(100000, 999999)),
        "REF" + str(random.randint(100000, 999999)),
        "TRN" + str(random.randint(100000, 999999)),
    ])


def _rand_link():
    domains = [
        "secure-{bank}.com", "{bank}-online.xyz", "verify-{bank}.top",
        "{bank}-login.xyz", "account-{bank}.tk", "secure-center.xyz",
        "portal-verify.com", "account-update.net", "security-check.xyz",
        "bank-verify.com",
    ]
    domain = random.choice(domains)
    return f"https://www.{random.choice(['gtbank.com', 'ubagroup.com', 'accessbankplc.com', 'zenithbank.com', 'fidelitybank.ng', 'firstbanknigeria.com', 'moniepoint.com', 'palmpay.com', 'opay.ng', 'kuda.com', 'carbon.co'])}/{random.choice(['reset-password', 'verify', 'statement', 'update-kyc', 'support'])}-{random.randint(1000, 9999)}"


def _rand_phone():
    prefixes = ["080", "081", "090", "070", "091"]
    return random.choice(prefixes) + "".join([str(random.randint(0, 9)) for _ in range(8)])


def _rand_email():
    return f"support@{random.choice(['bank-verify.xyz', 'account-update.com', 'secure-center.net', 'banking-portal.top'])}"


def _rand_atm_id():
    return f"ATM-{random.choice(BANKS[:5])}-{random.randint(100, 999)}"


def _gen_legit(amount):
    idx = random.randrange(len(LEGIT_TEMPLATES))
    t = LEGIT_TEMPLATES[idx]
    bank = random.choice(ORGS)
    text = t.format(
        customer=random.choice(CUSTOMER_NAMES),
        amount=f"{amount:,}",
        bank=bank,
        account=random.choice(ACCOUNT_NUMS),
        merchant=random.choice(MERCHANTS),
        date=_rand_date(),
        balance=f"{_rand_balance(amount):,}",
        phone=_rand_phone(),
        atm_id=_rand_atm_id(),
        ref=_rand_ref(),
        link=_rand_link(),
        sender=random.choice(CUSTOMER_NAMES),
        recipient=random.choice(CUSTOMER_NAMES),
        rate=str(random.randint(5, 25)),
        email=_rand_email(),
        count=str(random.randint(1, 5)),
    )[:512]
    return text, f"LEGIT_{idx:02d}"


def _gen_trad(amount):
    idx = random.randrange(len(TRAD_PHISH_TEMPLATES))
    t = TRAD_PHISH_TEMPLATES[idx]
    bank = random.choice(ORGS)
    shady = ["account-verify.tk", "secure-bank.top", "update-info.xyz",
             "bank-login.ml", "verify-account.ga", "secure-center.cf",
             "portal-update.xyz", "account-reactivation.tk"]
    link = f"https://{random.choice(shady)}/{random.randint(10000, 99999)}"
    text = t.format(
        customer=random.choice(CUSTOMER_NAMES),
        amount=f"{amount:,}",
        bank=bank,
        link=link,
        phone=_rand_phone(),
        email=_rand_email(),
        count=str(random.randint(1, 10)),
    )[:512]
    return text, f"TRAD_{idx:02d}"


def _gen_ai(amount):
    idx = random.randrange(len(AI_PHISH_TEMPLATES))
    t = AI_PHISH_TEMPLATES[idx]
    bank = random.choice(ORGS)
    ai_domains = [
        f"secure.{bank.lower().replace(' ', '')}-portal.com",
        f"verify.{bank.lower().replace(' ', '')}-online.ng",
        f"compliance.{bank.lower().replace(' ', '')}.org",
        f"account.{bank.lower().replace(' ', '')}-secure.net",
        f"portal.{bank.lower().replace(' ', '')}-verify.com",
    ]
    link = f"https://{random.choice(ai_domains)}/{random.choice(['verify', 'compliance', 'secure', 'update', 'confirm'])}-{random.randint(1000, 9999)}"
    text = t.format(
        customer=random.choice(CUSTOMER_NAMES),
        amount=f"{amount:,}",
        bank=bank,
        link=link,
        date=_rand_date(),
        phone=_rand_phone(),
        email=_rand_email(),
    )[:1024]
    return text, f"AI_{idx:02d}"


# ---- Generate 5000+ samples ----
samples = []
for _ in range(1750):
    text, tid = _gen_legit(_rand_amount(500, 500000))
    samples.append((text, 0, tid))
for _ in range(1750):
    text, tid = _gen_trad(_rand_amount())
    samples.append((text, 1, tid))
for _ in range(1750):
    text, tid = _gen_ai(_rand_amount())
    samples.append((text, 2, tid))
random.shuffle(samples)

seen, deduped = set(), []
for text, label, tid in samples:
    if text not in seen:
        seen.add(text)
        deduped.append((text, label, tid))

csv_path = '/content/dataset.csv'
meta_csv_path = '/content/dataset_metadata.csv'
os.makedirs('/content', exist_ok=True)
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['text', 'label'])
    w.writerows([(t, l) for t, l, _ in deduped])
with open(meta_csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['text', 'label', 'template_family_id'])
    w.writerows(deduped)

from collections import Counter
dist = Counter(label for _, label, _ in deduped)
tid_dist = Counter(tid for _, _, tid in deduped)
print(f"Total before dedup: {len(samples)} | After dedup: {len(deduped)}")
print(f"Class distribution: {dict(sorted(dist.items()))}")
print(f"Unique template families: {len(tid_dist)}")
print(f"Training split (80%) = {int(len(deduped) * 0.8)} samples")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('/content/dataset.csv')
df = df.dropna(subset=['text', 'label']).reset_index(drop=True)
df['label'] = df['label'].astype(int)
print(f'Total samples: {len(df)}')
print(f'Class distribution:\n{df["label"].value_counts().sort_index()}')

In [ ]:
# Split into train (80%), validation (10%), test (10%) — stratified
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

In [ ]:
# ============ TABLE 4.1: Dataset Composition ============
counts = df['label'].value_counts().sort_index()

print('='*60)
print('Table 4.1: Dataset Composition')
print('='*60)
print(f'{"Class":40s} {"Samples":>8s}')
print('-'*60)
for i in range(3):
    print(f'{LABEL_MAP[i]:40s} {counts[i]:>8d}')
print('-'*60)
print(f'{"Total":40s} {len(df):>8d}')
print('='*60)

In [ ]:
# ============ FIGURE 4.1: Dataset Class Distribution ============
labels = ['Legitimate\nFinancial Comm.', 'Traditional\nPhishing', 'AI-Generated\nPhishing']
plt.figure(figsize=(9, 5))
bars = plt.bar(labels, counts.values, color=CLASS_COLORS, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
             str(count), ha='center', va='bottom', fontweight='bold', fontsize=13)
plt.ylabel('Number of Samples', fontsize=12)
plt.title('Figure 4.1: Dataset Class Distribution', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/dataset_distribution.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: figures/dataset_distribution.png')

In [ ]:
from transformers import DistilBertTokenizerFast

MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 128
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize(texts):
    return tokenizer(texts, truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='np')

train_enc = tokenize(train_df['text'].tolist())
val_enc = tokenize(val_df['text'].tolist())
test_enc = tokenize(test_df['text'].tolist())
print('Tokenization complete')

In [ ]:
import torch
import transformers
transformers.logging.set_verbosity_error()
from transformers import DistilBertForSequenceClassification

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: 'Legitimate', 1: 'Traditional Phishing', 2: 'AI-Generated Phishing'},
    label2id={'Legitimate': 0, 'Traditional Phishing': 1, 'AI-Generated Phishing': 2},
)
model.to(device)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from transformers import get_scheduler
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             average_precision_score)
from sklearn.preprocessing import label_binarize

BATCH_SIZE = 16
EPOCHS = 5
LR = 2e-5

train_ds = TensorDataset(torch.tensor(train_enc['input_ids']),
                         torch.tensor(train_enc['attention_mask']),
                         torch.tensor(train_df['label'].values))
val_ds = TensorDataset(torch.tensor(val_enc['input_ids']),
                       torch.tensor(val_enc['attention_mask']),
                       torch.tensor(val_df['label'].values))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

optimizer = AdamW(model.parameters(), lr=LR)
num_steps = EPOCHS * len(train_loader)
scheduler = get_scheduler('linear', optimizer=optimizer,
                          num_warmup_steps=0, num_training_steps=num_steps)
print(f'Steps per epoch: {len(train_loader)} | Total steps: {num_steps}')

In [ ]:
import os
os.makedirs('/content/models', exist_ok=True)

best_f1 = 0.0
history = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    progress = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch in progress:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        progress.set_postfix({'loss': f'{loss.item():.4f}'})
    avg_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    preds_all, labels_all, probs_all = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids, attention_mask, labels = [b.to(device) for b in batch]
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=-1)
            preds_all.extend(torch.argmax(probs, dim=-1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
            probs_all.extend(probs.cpu().numpy())

    acc = accuracy_score(labels_all, preds_all)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_all, preds_all, average='weighted', zero_division=0)
    y_bin = label_binarize(labels_all, classes=[0, 1, 2])
    auprc = np.mean([average_precision_score(y_bin[:, i], np.array(probs_all)[:, i]) for i in range(3)])

    history.append({'epoch': epoch+1, 'loss': avg_loss, 'val_acc': acc, 'val_f1': f1, 'auprc': auprc})
    print(f'Epoch {epoch+1}: loss={avg_loss:.4f}, accuracy={acc:.4f}, precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}, auprc={auprc:.4f}')

    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained('/content/models/phishing_model')
        tokenizer.save_pretrained('/content/models/tokenizer')
        print(f'  -> New best model saved (F1={f1:.4f})')

# Print history for your figures
print()
print('TRAINING HISTORY FOR FIGURES:')
for h in history:
    print(f"Epoch {h['epoch']}: loss={h['loss']:.4f}, val_acc={h['val_acc']:.4f}, val_f1={h['val_f1']:.4f}, auprc={h['auprc']:.4f}")

In [ ]:
test_ds = TensorDataset(torch.tensor(test_enc['input_ids']),
                        torch.tensor(test_enc['attention_mask']),
                        torch.tensor(test_df['label'].values))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

model.eval()
preds_all, labels_all, probs_all = [], [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)
        preds_all.extend(torch.argmax(probs, dim=-1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
        probs_all.extend(probs.cpu().numpy())

print('='*50)
print('TABLE 4.4: MODEL PERFORMANCE (TEST SET)')
print('='*50)
acc = accuracy_score(labels_all, preds_all)
precision, recall, f1, _ = precision_recall_fscore_support(
    labels_all, preds_all, average='weighted', zero_division=0)
y_bin = label_binarize(labels_all, classes=[0, 1, 2])
auprc = np.mean([average_precision_score(y_bin[:, i], np.array(probs_all)[:, i]) for i in range(3)])
print(f'Accuracy:  {acc:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {f1:.4f}')
print(f'AUPRC:     {auprc:.4f}')
print()
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
names = ['Legitimate', 'Traditional Phishing', 'AI-Generated Phishing']
print(classification_report(labels_all, preds_all, target_names=names, zero_division=0))

import matplotlib.pyplot as plt
cm = confusion_matrix(labels_all, preds_all)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Confusion Matrix — DistilBERT (5000-sample dataset)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: /content/confusion_matrix.png')

# --- Text Confusion Matrix (verifiable without image) ---
label_w = max(len(l) for l in names)
cell_w = max(len(str(cm.max())), 6) + 2
col_hdrs = [f'P:{l}' for l in names]
row_hdrs = [f'T:{l}' for l in names]
sep = '+' + '-'*(label_w+2) + '+' + '+'.join('-'*cell_w for _ in range(3)) + '+'
print()
print('Confusion Matrix (text):')
print(sep)
print('|' + ' '*(label_w+2) + '|' + '|'.join(f'{h:^{cell_w}}' for h in col_hdrs) + '|')
print(sep.replace('-', '='))
for i in range(3):
    print(f'|{row_hdrs[i]:>{label_w+2}}|' + '|'.join(f'{cm[i][j]:^{cell_w}}' for j in range(3)) + '|')
print(sep)

off_diag = int(cm.sum() - np.trace(cm))
total = int(cm.sum())
print(f'\nOverall accuracy: {int(np.trace(cm))}/{total} = {np.trace(cm)/total:.4f}')
if off_diag == 0:
    print('CONFIRMED: Zero misclassifications across all classes.')
else:
    print(f'{off_diag} misclassification(s) found.')

print('\n' + '='*60)
print('CAVEAT: Perfect 1.0000 test-set scores should not be interpreted')
print('as evidence of guaranteed real-world performance. The dataset is')
print('procedurally generated from a small number of templates with minor')
print('slot substitutions. Only exact-string deduplication was applied,')
print('so near-duplicate or template-sibling samples may span both the')
print('train and test splits. Run the data-integrity validation below')
print('to check for near-duplicate contamination.')
print('='*60)

In [ ]:
# ============ FIGURES 4.3 & 4.4: Training Curves (real history) ============
epochs = [h['epoch'] for h in history]
losses = [h['loss'] for h in history]
accs = [h['val_acc'] for h in history]
f1s = [h['val_f1'] for h in history]

# Figure 4.3: Training Loss Curve
plt.figure(figsize=(8, 5))
plt.plot(epochs, losses, 'b-o', linewidth=2, markersize=8)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Figure 4.3: Training Loss Curve', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.xticks(epochs)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/training_loss_curve.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: figures/training_loss_curve.png')

# Figure 4.4: Validation Accuracy & F1 Curve
plt.figure(figsize=(8, 5))
plt.plot(epochs, accs, 'g-s', linewidth=2, markersize=8, label='Validation Accuracy')
plt.plot(epochs, f1s, 'm-d', linewidth=2, markersize=8, label='Validation F1 Score')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Figure 4.4: Validation Accuracy & F1 Curve', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.xticks(epochs)
plt.ylim(0.5, 1.0)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/validation_accuracy_curve.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: figures/validation_accuracy_curve.png')

In [ ]:
# ============ TABLE 4.6: Inference Latency ============
times_ms = []
sample_texts = test_df['text'].sample(100, random_state=42).tolist()

for text in sample_texts:
    enc = tokenizer(text, truncation=True, padding='max_length',
                    max_length=MAX_LEN, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(**enc)
    elapsed = (time.perf_counter() - start) * 1000
    times_ms.append(elapsed)

times = np.array(times_ms)
print()
print('='*50)
print('Table 4.6: Inference Latency')
print('='*50)
print(f'{"Metric":25s} {"Value":>15s}')
print('-'*50)
print(f'{"Average Response Time":25s} {times.mean():>10.2f} ms')
print(f'{"Fastest Response":25s} {times.min():>10.2f} ms')
print(f'{"Slowest Response":25s} {times.max():>10.2f} ms')
print(f'{"Std Deviation":25s} {times.std():>10.2f} ms')
print(f'{"Samples Tested":25s} {len(times):>10d}')
print('='*50)

In [ ]:
# ============ TABLE 4.5: Regex Baseline Comparison ============
def regex_classify(text):
    t = text.lower()
    ai_score = 0
    ai_patterns = ['central bank of nigeria', 'cbn directive', 'regulatory compliance',
                   'beneficial ownership', 'anti-money laundering', 'know your customer',
                   'aml', 'kyc', 'compliance notice', 'mandatory']
    for p in ai_patterns:
        if p in t: ai_score += 1

    trad_score = 0
    trad_patterns = ['urgent', 'click here', 'suspended', 'blocked',
                     'verify.*account', 'bvn.*block', 'reactivate',
                     'immediate', 'warning', 'expired']
    for p in trad_patterns:
        if re.search(p, t): trad_score += 1

    legit_score = 0
    legit_patterns = ['debit alert', 'credit alert', 'available balance',
                      'transaction alert', 'monthly statement', 'account balance',
                      'deposit', 'withdrawal']
    for p in legit_patterns:
        if p in t: legit_score += 1

    if ai_score > trad_score and ai_score > legit_score:
        return 2
    elif trad_score > legit_score:
        return 1
    return 0

y_true = np.array(labels_all)
y_pred = np.array(preds_all)
texts = test_df['text'].tolist()

y_pred_regex = np.array([regex_classify(t) for t in texts])

acc_regex = accuracy_score(y_true, y_pred_regex)
p_regex, r_regex, f_regex, _ = precision_recall_fscore_support(
    y_true, y_pred_regex, average='weighted', zero_division=0)

acc_model = accuracy_score(y_true, y_pred)
p_model, r_model, f_model, _ = precision_recall_fscore_support(
    y_true, y_pred, average='weighted', zero_division=0)

print()
print('='*65)
print('Table 4.5: Comparison with Regex Baseline')
print('='*65)
print(f'{"Metric":12s} {"Regex-Based":>16s} {"Proposed":>16s} {"Improvement":>14s}')
print('-'*65)
print(f'{"Accuracy":12s} {acc_regex:>10.4f}      {acc_model:>10.4f}      {(acc_model-acc_regex):>+8.4f}')
print(f'{"Precision":12s} {p_regex:>10.4f}      {p_model:>10.4f}      {(p_model-p_regex):>+8.4f}')
print(f'{"Recall":12s} {r_regex:>10.4f}      {r_model:>10.4f}      {(r_model-r_regex):>+8.4f}')
print(f'{"F1-Score":12s} {f_regex:>10.4f}      {f_model:>10.4f}      {(f_model-f_regex):>+8.4f}')
print('='*65)

In [ ]:
# ============ TABLE 4.7: Zero-Shot Detection Results ============
scenarios = [
    ('Scenario 1: Legitimate Transaction',
     'Dear Chidi, a debit of NGN25,000 was made on your GTBank account '
     '0123456789 at ShopRite on 15-Jan-2025. Available balance: NGN450,000.'),
    ('Scenario 2: Traditional Phishing',
     'URGENT!!! Your BVN has been BLOCKED! Click here to verify now: '
     'https://account-verify.tk/38472'),
    ('Scenario 3: AI-Generated Phishing',
     'Subject: Mandatory BVN-NIN Linkage Compliance Notice\n\n'
     'Dear Customer, the Central Bank of Nigeria (CBN) requires all accounts '
     'to have BVN linked to NIN by 30-Mar-2025. Non-compliant accounts will '
     'be restricted. Complete linkage: https://secure-gtbank-portal.com/verify'),
]

expected_labels = ['Legitimate Financial Comm.', 'Traditional Phishing', 'AI-Generated Phishing']
predicted_labels = ['Legitimate Financial Comm.', 'Traditional Phishing', 'AI-Generated Phishing']

print()
print('='*90)
print('Table 4.7: Zero-Shot Detection Results')
print('='*90)
print(f'{"Test Scenario":30s} {"Expected":25s} {"Predicted":25s} {"Status":>8s}')
print('-'*90)

for i, (name, text) in enumerate(scenarios):
    enc = tokenizer(text, truncation=True, padding='max_length',
                    max_length=MAX_LEN, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        outputs = model(**enc)
    pred = torch.argmax(outputs.logits, dim=-1).item()
    status = 'Correct' if pred == i else 'Incorrect'
    print(f'{name:30s} {expected_labels[i]:25s} {predicted_labels[pred]:25s} {status:>8s}')
print('='*90)

In [ ]:
# ============ TABLE 4.3: Hyperparameter Configuration ============
print()
print('='*50)
print('Table 4.3: Hyperparameter Configuration')
print('='*50)
print(f'{"Parameter":22s} {"Value":>25s}')
print('-'*50)
print(f'{"Pre-trained Model":22s} {MODEL_NAME:>25s}')
print(f'{"Max Sequence Length":22s} {MAX_LEN:>25d}')
print(f'{"Learning Rate":22s} {LR:>25.0e}')
print(f'{"Batch Size":22s} {BATCH_SIZE:>25d}')
print(f'{"Epochs":22s} {EPOCHS:>25d}')
print(f'{"Optimizer":22s} {"AdamW":>25s}')
print(f'{"Loss Function":22s} {"Cross Entropy":>25s}')
print(f'{"Scheduler":22s} {"Linear (no warmup)":>25s}')
print(f'{"Weight Decay":22s} {"0.01":>25s}')
print(f'{"Dataset Size":22s} {len(df):>25d}')
print(f'{"Train / Val / Test":22s} {f"{len(train_df)}/{len(val_df)}/{len(test_df)}":>25s}')
print('='*50)

In [ ]:
# ============ FIGURE 4.5: SHAP Explanation (AI-Generated Phishing) ============
# SHAP is MANDATORY: a failure here fails the notebook (no silent skip).
import shap
test_text = (
    'Subject: Mandatory BVN-NIN Linkage Compliance Notice\n\n'
    'Dear Chidi Okonkwo,\n\n'
    'This is to notify you that the Central Bank of Nigeria (CBN) now requires '
    'all bank accounts to have their BVN linked to the National Identification '
    'Number (NIN) by 30-Mar-2025. Accounts not complying will be placed on '
    'restricted status.\n\n'
    'To complete the linkage securely, please visit: '
    'https://secure.gtbank-portal.com/verify-3847\n\n'
    'Thank you for your cooperation.\n'
    'Compliance Department\n'
    'GTBank'
)

def predict_proba_batch(texts):
    if isinstance(texts, str):
        texts = [texts]
    elif isinstance(texts, np.ndarray):
        texts = texts.tolist()
    texts = list(texts)
    texts = [t if isinstance(t, str) else ' '.join(t) for t in texts]
    enc = tokenizer(texts, truncation=True, padding='max_length',
                    max_length=MAX_LEN, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()

masker = shap.maskers.Text(tokenizer, mask_token='...', collapse_mask_token=True)
explainer = shap.Explainer(predict_proba_batch, masker, output_names=CLASS_NAMES, seed=42)

enc_test = tokenizer(test_text, truncation=True, padding='max_length',
                     max_length=MAX_LEN, return_tensors='pt')
enc_test = {k: v.to(device) for k, v in enc_test.items()}
with torch.no_grad():
    logits = model(**enc_test).logits
pred_class = torch.argmax(logits, dim=-1).item()

print('Running SHAP explanation (30-60s on GPU)...')
shap_values = explainer([test_text], max_evals=50, batch_size=1)

for class_idx in range(3):
    plt.figure()
    shap.waterfall_plot(shap_values[0, :, class_idx], show=False, max_display=15)
    plt.title(f'SHAP Waterfall - Class: {CLASS_NAMES[class_idx]}', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/shap_waterfall_class_{class_idx}.png', dpi=200, bbox_inches='tight')
    plt.show()

words = tokenizer.convert_ids_to_tokens(
    tokenizer(test_text, truncation=True, padding='max_length', max_length=MAX_LEN)['input_ids'])
vals = shap_values[0, :, pred_class].values
top_indices = np.argsort(np.abs(vals))[-10:]

print(f'\nTop 10 influential words (for {CLASS_NAMES[pred_class]}):')
print(f'{"Word":20s} {"SHAP Value":>10s} {"Impact"}')
print('-'*45)
for idx in reversed(top_indices):
    word = words[idx]
    if word in ['[PAD]', '[CLS]', '[SEP]']:
        continue
    val = vals[idx]
    impact = 'AI Phish' if val > 0.1 else ('Trad Phish' if val > 0.05 else ('Legit' if val < -0.05 else 'Neutral'))
    print(f'{word:20s} {val:>10.4f}  {impact}')
print('Saved SHAP waterfall plots to figures/')


In [ ]:
# ============ DATA INTEGRITY VALIDATION ============
# Checks for near-duplicate contamination in train/test split.
# Run this BEFORE packaging artifacts.

import hashlib
from collections import defaultdict

SLOT_RE = re.compile(
    r'NGN[\d,]+|\b\d{9,15}\b|\b\d{2}-[A-Z]{3}-\d{4}\b|'
    r'\b\d{4}-\d{2}-\d{2}\b|https?://\S+|\b0\d{10}\b|\*\d{3,4}\#'
)

def _skeleton(text):
    t = re.sub(r'\s+', ' ', text.lower()).strip()
    t = SLOT_RE.sub(' <SLOT> ', t)
    return re.sub(r'\s+', ' ', t).strip()

def _ngrams(text, n=3):
    t = re.sub(r'\s+', ' ', text.lower()).strip()
    return set(t[i:i+n] for i in range(max(len(t)-n+1, 1)))

# AI-phishing only (class 2)
ai_train = train_df[train_df['label']==2]['text'].tolist()
ai_test  = test_df[test_df['label']==2]['text'].tolist()
print(f'AI-phishing samples: train={len(ai_train)}, test={len(ai_test)}')

# (i) N-gram Jaccard
tr_ng = [_ngrams(t) for t in ai_train]
te_ng = [_ngrams(t) for t in ai_test]
max_j, pair = 0.0, (-1,-1)
above_80 = 0
for i, a in enumerate(tr_ng):
    for j, b in enumerate(te_ng):
        s = len(a & b) / len(a | b) if (a | b) else 1.0
        if s > max_j:
            max_j, pair = s, (i,j)
        if s >= 0.80:
            above_80 += 1
print(f'\nN-gram Jaccard: max={max_j:.4f}, pairs>=0.80={above_80}')

# (ii) Template families
train_sk = [_skeleton(t) for t in ai_train]
test_sk  = [_skeleton(t) for t in ai_test]
train_hashes = [hashlib.sha256(s.encode()).hexdigest()[:16] for s in train_sk]
test_hashes  = [hashlib.sha256(s.encode()).hexdigest()[:16] for s in test_sk]
shared = set(train_hashes) & set(test_hashes)
test_in_shared = sum(test_hashes.count(h) for h in shared)
contam = test_in_shared / len(ai_test) if ai_test else 0
print(f'\nTemplate families: total={len(set(train_hashes)|set(test_hashes))}, '
      f'shared={len(shared)}, test_contamination={contam:.1%}')

# Verdict
print('\n' + '='*50)
if above_80 > 0 or len(shared) > 0:
    print('WARNING: Potential near-duplicate contamination found.')
    print('The perfect 1.0000 scores may be explained by this.')
    print('Recommend: group-based split or external test set.')
else:
    print('OK: No near-duplicate contamination detected.')
print('='*50)

In [ ]:
# ============ CAPTURE ORIGINAL RESULTS (disk, verifiable) ============
# Snapshot original random-split results to disk before the template-aware
# re-run overwrites any variables. Everything is saved, not only printed.

orig_metrics = {'accuracy': float(acc), 'precision': float(precision),
                'recall': float(recall), 'f1': float(f1), 'auprc': float(auprc)}
orig_cm = np.array(cm).tolist()
orig_history = [dict(h) for h in history]

orig_tables = {
    'dataset_composition': {str(int(k)): int(v) for k, v in df['label'].value_counts().sort_index().items()},
    'splits': {'train': int(len(train_df)), 'val': int(len(val_df)), 'test': int(len(test_df))},
    'latency_ms': {'mean': float(times.mean()), 'min': float(times.min()),
                   'max': float(times.max()), 'std': float(times.std()), 'samples': int(len(times))},
    'regex_baseline': {'accuracy': float(acc_regex), 'precision': float(p_regex),
                       'recall': float(r_regex), 'f1': float(f_regex)},
    'zero_shot': [],
}
for i, (name, text) in enumerate(scenarios):
    enc = tokenizer(text, truncation=True, padding='max_length',
                    max_length=MAX_LEN, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        pred = torch.argmax(model(**enc).logits, dim=-1).item()
    orig_tables['zero_shot'].append({'scenario': name, 'expected': expected_labels[i],
                                   'predicted': predicted_labels[pred], 'correct': bool(pred == i)})

with open('/content/original_results.json', 'w') as f:
    json.dump({'metrics': orig_metrics, 'confusion_matrix': orig_cm,
               'history': orig_history, 'tables': orig_tables}, f, indent=2, default=str)
print('Saved: /content/original_results.json')
print('Captured original metrics:', {k: round(v, 4) for k, v in orig_metrics.items()})


In [ ]:
# ============ TEMPLATE-AWARE GROUP SPLIT (zero template overlap) ============
from collections import defaultdict

meta_df = pd.read_csv('/content/dataset_metadata.csv')
meta_df = meta_df.dropna(subset=['text', 'label', 'template_family_id']).reset_index(drop=True)
meta_df['label'] = meta_df['label'].astype(int)
meta_df['template_family_id'] = meta_df['template_family_id'].astype(str)
print(f'Metadata samples: {len(meta_df)} | template families: {meta_df["template_family_id"].nunique()}')
print(f'Class distribution:\n{meta_df["label"].value_counts().sort_index()}')

def group_based_split(mdf, seed=42):
    rng = random.Random(seed)
    class_families = defaultdict(list)
    for tid, grp in mdf.groupby('template_family_id'):
        class_families[grp['label'].iloc[0]].append(tid)
    train_fams, val_fams, test_fams = set(), set(), set()
    for label in sorted(class_families):
        families = list(class_families[label])
        rng.shuffle(families)
        total = len(families)
        n_train = max(1, round(total * 0.80))
        n_val = max(1, round(total * 0.10))
        n_test = total - n_train - n_val
        if n_test < 1:
            n_test = 1
            n_train = total - n_val - n_test
        if n_val < 1:
            n_val = 1
            n_train = total - n_val - n_test
        train_fams.update(families[:n_train])
        val_fams.update(families[n_train:n_train+n_val])
        test_fams.update(families[n_train+n_val:])
        print(f'  Class {label}: {n_train} train / {n_val} val / {n_test} test families')
    assert not (train_fams & val_fams), "train/val overlap"
    assert not (train_fams & test_fams), "train/test overlap"
    assert not (val_fams & test_fams), "val/test overlap"
    def sel(fams): return mdf[mdf['template_family_id'].isin(fams)].reset_index(drop=True)
    return {'train': sel(train_fams), 'validation': sel(val_fams), 'test': sel(test_fams)}

print("Performing template-aware group-based split ...")
ta_splits = group_based_split(meta_df)
ta_train_df, ta_val_df, ta_test_df = ta_splits['train'], ta_splits['validation'], ta_splits['test']

ta_fams = {k: set(v["template_family_id"].unique()) for k, v in ta_splits.items()}
print()
print("TEMPLATE-AWARE SPLIT SIZES:")
print(f"  Train = {len(ta_train_df)} | Val = {len(ta_val_df)} | Test = {len(ta_test_df)}")
print(f"  Shared train/val families = {len(ta_fams['train'] & ta_fams['validation'])}")
print(f"  Shared train/test families = {len(ta_fams['train'] & ta_fams['test'])}")
print(f"  Shared val/test families = {len(ta_fams['validation'] & ta_fams['test'])}")
for name, sdf in ta_splits.items():
    dist_s = sdf['label'].value_counts().sort_index().to_dict()
    print(f"  {name.upper()}: samples={len(sdf)}, classes={dist_s}, families={len(set(sdf['template_family_id']))}")


In [ ]:
# ============ TEMPLATE-AWARE: RETRAIN DISTILBERT (same hyperparameters) ============
# Free the original model from memory; its saved files remain under /content/models.
del model
torch.cuda.empty_cache()

ta_device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Template-aware model device: {ta_device}")

ta_model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3,
    id2label={0: 'Legitimate', 1: 'Traditional Phishing', 2: 'AI-Generated Phishing'},
    label2id={'Legitimate': 0, 'Traditional Phishing': 1, 'AI-Generated Phishing': 2},
)
ta_model.to(ta_device)

def load_and_tokenize_ta(sdf):
    sdf = sdf.dropna(subset=['text', 'label']).reset_index(drop=True)
    sdf['label'] = sdf['label'].astype(int)
    enc = tokenizer(sdf['text'].tolist(), truncation=True, padding='max_length',
                    max_length=MAX_LEN, return_tensors='np')
    return TensorDataset(torch.tensor(enc['input_ids']), torch.tensor(enc['attention_mask']),
                         torch.tensor(sdf['label'].values))

ta_train_ds = load_and_tokenize_ta(ta_train_df)
ta_val_ds = load_and_tokenize_ta(ta_val_df)
ta_test_ds = load_and_tokenize_ta(ta_test_df)
ta_train_loader = DataLoader(ta_train_ds, batch_size=BATCH_SIZE, shuffle=True)
ta_val_loader = DataLoader(ta_val_ds, batch_size=BATCH_SIZE)
ta_test_loader = DataLoader(ta_test_ds, batch_size=BATCH_SIZE)
print(f"Train: {len(ta_train_df)} | Val: {len(ta_val_df)} | Test: {len(ta_test_df)}")

os.makedirs('/content/template_aware_model/phishing_model', exist_ok=True)
os.makedirs('/content/template_aware_model/tokenizer', exist_ok=True)

ta_optimizer = AdamW(ta_model.parameters(), lr=LR)
ta_num_steps = EPOCHS * len(ta_train_loader)
ta_scheduler = get_scheduler('linear', optimizer=ta_optimizer,
                               num_warmup_steps=0, num_training_steps=ta_num_steps)

ta_best_f1 = 0.0
ta_history = []
for epoch in range(EPOCHS):
    ta_model.train()
    total_loss = 0
    progress = tqdm(ta_train_loader, desc=f'TA Epoch {epoch+1}/{EPOCHS}')
    for batch in progress:
        input_ids, attention_mask, labels = [b.to(ta_device) for b in batch]
        outputs = ta_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        ta_optimizer.step()
        ta_scheduler.step()
        ta_optimizer.zero_grad()
        progress.set_postfix({'loss': f'{loss.item():.4f}'})
    avg_loss = total_loss / len(ta_train_loader)

    ta_model.eval()
    vp, vl, vpr = [], [], []
    with torch.no_grad():
        for batch in ta_val_loader:
            input_ids, attention_mask, labels = [b.to(ta_device) for b in batch]
            probs = torch.softmax(ta_model(input_ids=input_ids, attention_mask=attention_mask).logits, dim=-1)
            vp.extend(torch.argmax(probs, dim=-1).cpu().numpy())
            vl.extend(labels.cpu().numpy())
            vpr.extend(probs.cpu().numpy())
    vacc = accuracy_score(vl, vp)
    vp_, vr_, vf1, _ = precision_recall_fscore_support(vl, vp, average='weighted', zero_division=0)
    vy_bin = label_binarize(vl, classes=[0, 1, 2])
    vauprc = np.mean([average_precision_score(vy_bin[:, i], np.array(vpr)[:, i]) for i in range(3)])
    ta_history.append({'epoch': epoch+1, 'loss': avg_loss, 'val_acc': vacc, 'val_f1': vf1, 'auprc': vauprc})
    print(f'TA Epoch {epoch+1}: loss={avg_loss:.4f}, acc={vacc:.4f}, f1={vf1:.4f}, auprc={vauprc:.4f}')
    if vf1 > ta_best_f1:
        ta_best_f1 = vf1
        ta_model.save_pretrained('/content/template_aware_model/phishing_model')
        tokenizer.save_pretrained('/content/template_aware_model/tokenizer')
        print(f'  -> Saved TA model (F1={vf1:.4f})')

print("Template-aware training complete.")


In [ ]:
# ============ TEMPLATE-AWARE: EVALUATION ON HELD-OUT TEST SET ============
ta_model.eval()
ta_preds, ta_labels, ta_probs = [], [], []
with torch.no_grad():
    for batch in ta_test_loader:
        input_ids, attention_mask, labels = [b.to(ta_device) for b in batch]
        probs = torch.softmax(ta_model(input_ids=input_ids, attention_mask=attention_mask).logits, dim=-1)
        ta_preds.extend(torch.argmax(probs, dim=-1).cpu().numpy())
        ta_labels.extend(labels.cpu().numpy())
        ta_probs.extend(probs.cpu().numpy())

ta_y_true = np.array(ta_labels)
ta_y_pred = np.array(ta_preds)
ta_y_prob = np.array(ta_probs)

ta_acc = accuracy_score(ta_y_true, ta_y_pred)
ta_precision, ta_recall, ta_f1, _ = precision_recall_fscore_support(
    ta_y_true, ta_y_pred, average='weighted', zero_division=0)
y_bin_ta = label_binarize(ta_y_true, classes=[0, 1, 2])
ta_auprc = np.mean([average_precision_score(y_bin_ta[:, i], ta_y_prob[:, i]) for i in range(3)])

ta_metrics = {'accuracy': round(float(ta_acc), 4), 'precision': round(float(ta_precision), 4),
              'recall': round(float(ta_recall), 4), 'f1': round(float(ta_f1), 4), 'auprc': round(float(ta_auprc), 4)}

print('='*60)
print('TEMPLATE-AWARE SPLIT — TEST SET RESULTS')
print(f"Test samples: {len(ta_test_df)}")
print('='*60)
print(f'Accuracy:  {ta_metrics["accuracy"]:.4f}')
print(f'Precision: {ta_metrics["precision"]:.4f}')
print(f'Recall:    {ta_metrics["recall"]:.4f}')
print(f'F1-Score:  {ta_metrics["f1"]:.4f}')
print(f'AUPRC:     {ta_metrics["auprc"]:.4f}')
print()
print(classification_report(ta_y_true, ta_y_pred, target_names=CLASS_NAMES, zero_division=0))

ta_cm = confusion_matrix(ta_y_true, ta_y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(ta_cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Confusion Matrix — Template-Aware Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/template_aware_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: figures/template_aware_confusion_matrix.png")

# Text confusion matrix (verifiable without an image)
label_w = max(len(l) for l in CLASS_NAMES)
cell_w = max(len(str(ta_cm.max())), 6) + 2
col_hdrs = [f'P:{l}' for l in CLASS_NAMES]
row_hdrs = [f'T:{l}' for l in CLASS_NAMES]
sep = '+' + '-'*(label_w+2) + '+' + '+'.join('-'*cell_w for _ in range(3)) + '+'
print('\nTemplate-Aware Confusion Matrix (text):')
print(sep)
print('|' + ' '*(label_w+2) + '|' + '|'.join(f'{h:^{cell_w}}' for h in col_hdrs) + '|')
print(sep.replace('-', '='))
for i in range(3):
    print(f'|{row_hdrs[i]:>{label_w+2}}|' + '|'.join(f'{ta_cm[i][j]:^{cell_w}}' for j in range(3)) + '|')
print(sep)
print(f"\nCorrect: {int(np.trace(ta_cm))}/{ta_cm.sum()} = {np.trace(ta_cm)/ta_cm.sum():.4f}")
print(f"Misclassified: {ta_cm.sum() - int(np.trace(ta_cm))}")


In [ ]:
# ============ TEMPLATE-AWARE: TRAINING CURVES ============
ta_epochs = [h['epoch'] for h in ta_history]

plt.figure(figsize=(8, 5))
plt.plot(ta_epochs, [h['loss'] for h in ta_history], 'b-o', linewidth=2, markersize=8)
plt.xlabel('Epoch', fontsize=12); plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss — Template-Aware Split', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3); plt.xticks(ta_epochs); plt.tight_layout()
plt.savefig(f'{FIG_DIR}/template_aware_training_loss_curve.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: figures/template_aware_training_loss_curve.png")

plt.figure(figsize=(8, 5))
plt.plot(ta_epochs, [h['val_acc'] for h in ta_history], 'g-s', linewidth=2, markersize=8, label='Validation Accuracy')
plt.plot(ta_epochs, [h['val_f1'] for h in ta_history], 'm-d', linewidth=2, markersize=8, label='Validation F1')
plt.xlabel('Epoch', fontsize=12); plt.ylabel('Score', fontsize=12)
plt.title('Validation Accuracy & F1 — Template-Aware Split', fontsize=14, fontweight='bold')
plt.legend(fontsize=11); plt.grid(alpha=0.3); plt.xticks(ta_epochs); plt.ylim(0.0, 1.05); plt.tight_layout()
plt.savefig(f'{FIG_DIR}/template_aware_validation_curve.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: figures/template_aware_validation_curve.png")

print("Template-aware epoch-by-epoch history:")
for h in ta_history:
    print(f"  Epoch {h['epoch']}: loss={h['loss']:.4f}, val_acc={h['val_acc']:.4f}, val_f1={h['val_f1']:.4f}, auprc={h['auprc']:.4f}")


In [ ]:
# ============ TEMPLATE-AWARE: INTEGRITY VERIFICATION ============
# 1) Zero shared template families across splits (machine-checked).
family_overlap = {
    'train_val': len(ta_fams['train'] & ta_fams['validation']),
    'train_test': len(ta_fams['train'] & ta_fams['test']),
    'val_test': len(ta_fams['validation'] & ta_fams['test']),
}
print("Shared family verification (must all be OK):")
for k, v in family_overlap.items():
    print(f"  {k}: {v} [{'OK' if v == 0 else 'FAIL'}]")
assert family_overlap['train_test'] == 0, 'Template-aware split has family OVERLAP!'

# 2) Near-duplicate n-gram Jaccard between TA train and test.
def _ngrams_ta(t, n=3):
    t = re.sub(r'\s+', ' ', t.lower()).strip()
    return set(t[i:i+n] for i in range(max(len(t)-n+1, 1)))

def _jaccard_ta(a, b):
    return len(a & b) / len(a | b) if (a | b) else 1.0

ta_tr_ng = [_ngrams_ta(t) for t in ta_train_df['text'].tolist()]
ta_te_ng = [_ngrams_ta(t) for t in ta_test_df['text'].tolist()]
ta_max_j, ta_above80 = 0.0, 0
for i, a in enumerate(ta_tr_ng):
    for j, b in enumerate(ta_te_ng):
        s = _jaccard_ta(a, b)
        if s > ta_max_j: ta_max_j = s
        if s >= 0.80: ta_above80 += 1
print(f"TA train/test n-gram Jaccard: max={ta_max_j:.4f}, pairs>=0.80={ta_above80}")


In [ ]:
# ============ TEMPLATE-AWARE: COMPARISON & RESULTS CAPTURE ============
print('='*70)
print('COMPARISON: Original Random Split vs Template-Aware Split')
print('='*70)
header = f"{'Metric':<12} {'Original':>14} {'Template-Aware':>16}"
print(header)
print('-'*50)
for key in ['accuracy', 'precision', 'recall', 'f1', 'auprc']:
    print(f'{key.upper():<12} {orig_metrics[key]:>14.4f} {ta_metrics[key]:>16.4f}')
print('='*70)
print(f"\n  Original test set: {len(test_df)} samples")
print(f"  Template-aware test set: {len(ta_test_df)} samples")

# Classification reports saved to disk (verifiable):
with open('/content/original_classification_report.txt', 'w') as f:
    f.write(classification_report(np.array(labels_all), np.array(preds_all), target_names=CLASS_NAMES, zero_division=0))
with open('/content/template_aware_classification_report.txt', 'w') as f:
    f.write(classification_report(ta_y_true, ta_y_pred, target_names=CLASS_NAMES, zero_division=0))

results_summary = {
    'original_random_split': {'metrics': orig_metrics, 'splits': orig_tables['splits'],
                             'confusion_matrix': orig_cm, 'history': orig_history, 'tables': orig_tables},
    'template_aware_split': {'metrics': ta_metrics,
                            'train_size': int(len(ta_train_df)), 'val_size': int(len(ta_val_df)), 'test_size': int(len(ta_test_df)),
                            'class_distribution': {str(int(k)): int(v) for k, v in ta_test_df['label'].value_counts().sort_index().items()},
                            'family_overlap': family_overlap,
                            'n_gram_jaccard': {'max': round(float(ta_max_j), 6), 'pairs_ge_0.80': int(ta_above80)},
                            'confusion_matrix': ta_cm.tolist(), 'history': ta_history},
    'comparison': {k: {'original': orig_metrics[k], 'template_aware': ta_metrics[k]}
                   for k in ['accuracy', 'precision', 'recall', 'f1', 'auprc']},
}

with open('/content/results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2, default=str)
print('Saved: /content/results_summary.json (all metrics, matrices, overlap proof, history)')

print('\nACADEMIC INTEGRITY NOTE:')
print('  The template-aware split guarantees ZERO shared template families')
print('  between train/val/test (verified above). Differences from the')
print('  original 1.0000 results reflect true generalization to unseen templates.')


In [ ]:
# ============ PACKAGE & DOWNLOAD ALL ARTIFACTS ============
# 1) evidence_figures.zip - ALL PNGs (original + template-aware) for the write-up
with zipfile.ZipFile('/content/evidence_figures.zip', 'w') as z:
    for f in sorted(Path(FIG_DIR).glob('*.png')):
        z.write(f, arcname=f.name)
        print(f'Added: {f.name}')

# 2) results_json.zip - captured results, verifiable on disk
with zipfile.ZipFile('/content/results_json.zip', 'w') as z:
    z.write('/content/original_results.json', arcname='original_results.json')
    z.write('/content/results_summary.json', arcname='results_summary.json')
    z.write('/content/original_classification_report.txt', arcname='original_classification_report.txt')
    z.write('/content/template_aware_classification_report.txt', arcname='template_aware_classification_report.txt')
    z.write('/content/dataset_metadata.csv', arcname='dataset_metadata.csv')
    z.write('/content/dataset.csv', arcname='dataset.csv')
    for f in sorted(Path(FIG_DIR).glob('template_aware_*.png')):
        z.write(f, arcname=f'figures/{f.name}')

# 3) template-aware model + tokenizer
shutil.make_archive('/content/template_aware_model', 'zip', '/content/template_aware_model')

print('\nDownloading evidence_figures.zip ...')
files.download('/content/evidence_figures.zip')
print('Downloading results_json.zip ...')
files.download('/content/results_json.zip')
print('Downloading template_aware_model.zip ...')
files.download('/content/template_aware_model.zip')

# 4) original model + tokenizer zips
shutil.make_archive('/content/phishing_model', 'zip', '/content/models/phishing_model')
shutil.make_archive('/content/tokenizer', 'zip', '/content/models/tokenizer')

print('Downloading phishing_model.zip ...')
files.download('/content/phishing_model.zip')
print('Downloading tokenizer.zip ...')
files.download('/content/tokenizer.zip')
print('\nAll done. Backup copies also in the left file browser: /content/')
